# 04 Build User Sequences And Labels

Цель этапа: собрать пользовательские последовательности, сделать split по пользователям и посчитать labels для `retention_7d` / `retention_14d`.

Вход:

- `data/interim/events_normalized_*.parquet`
- `artifacts/vocab/event_token_vocab.json`

Выход:

- `data/splits/users_train.parquet`
- `data/splits/users_valid.parquet`
- `data/splits/users_test.parquet`
- `data/processed/train_prefixes.parquet`
- `data/processed/valid_prefixes.parquet`
- `data/processed/test_prefixes.parquet`
- `artifacts/manifests/sequences_manifest.json`

Label definition фиксируется здесь: `retention_Xd = 1`, если есть событие в окне `[prefix_end_ts + X days, prefix_end_ts + X days + 1 day)`.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

In [3]:
from src.sequences import build_user_sequences_and_labels, load_sequence_config
from src.tokenization import load_manifest

config = load_sequence_config(PROJECT_ROOT / "configs" / "sequences.yaml")
normalize_manifest = load_manifest(PROJECT_ROOT / config.input_manifest_path)

print("normalized shards:", normalize_manifest["num_shards"])
print("normalized rows:", normalize_manifest["rows"])
print("prefix lengths:", config.prefix_lengths)
print("retention days:", config.retention_days)
config

normalized shards: 74
normalized rows: 73063802
prefix lengths: (50, 100, 150)
retention days: (7, 14)


SequenceConfig(input_manifest_path='artifacts/manifests/normalize_manifest.json', vocab_path='artifacts/vocab/event_token_vocab.json', manifest_path='artifacts/manifests/sequences_manifest.json', work_dir='artifacts/work', sorted_events_path='artifacts/work/events_sorted_for_sequences.parquet', splits_dir='data/splits', processed_dir='data/processed', prefix_lengths=(50, 100, 150), retention_days=(7, 14), split_ratios={'train': 0.7, 'valid': 0.15, 'test': 0.15}, split_seed=42, duckdb_memory_limit='20GB', duckdb_threads=4, duckdb_temp_directory='artifacts/work/duckdb_tmp', compression='zstd', row_group_size=100000)

## Control Cell

Сначала выполните `DRY_RUN = True`: будет использован первый shard. Для полного прогона поставьте `DRY_RUN = False`.

`OVERWRITE_SORTED = False` позволяет не пересортировывать события, если sorted parquet уже создан. Если меняли входные данные или vocab, поставьте `True`.

In [4]:
DRY_RUN = False
MAX_SHARDS = 1 if DRY_RUN else None
OVERWRITE_SORTED = True if DRY_RUN else False

print("DRY_RUN:", DRY_RUN)
print("MAX_SHARDS:", MAX_SHARDS)
print("OVERWRITE_SORTED:", OVERWRITE_SORTED)

DRY_RUN: False
MAX_SHARDS: None
OVERWRITE_SORTED: False


In [5]:
manifest = build_user_sequences_and_labels(
    config=config,
    project_root=PROJECT_ROOT,
    max_shards=MAX_SHARDS,
    overwrite_sorted=OVERWRITE_SORTED,
)

manifest

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Build user prefixes:   0%|          | 0/729 [00:00<?, ?it/s]

{'created_at': '2026-05-10T12:56:36.661302+00:00',
 'input_manifest_path': 'artifacts/manifests/normalize_manifest.json',
 'vocab_path': 'artifacts/vocab/event_token_vocab.json',
 'sorted_events_path': 'artifacts/work/events_sorted_for_sequences.parquet',
 'max_shards': None,
 'dataset_max_ts': 1774814007,
 'prefix_lengths': [50, 100, 150],
 'retention_days': [7, 14],
 'label_definition': 'retention_Xd = any event in [prefix_end_ts + X days, prefix_end_ts + X days + 1 day); unavailable if dataset ends before window end',
 'time_gap_vocab_path': 'artifacts/vocab/time_gap_vocab.json',
 'splits': {'train': {'users': 47957, 'prefixes': 97122},
  'valid': {'users': 10317, 'prefixes': 21148},
  'test': {'users': 10270, 'prefixes': 20853}}}

In [6]:
import pandas as pd

split_summary = pd.DataFrame([
    {"split": split, **stats}
    for split, stats in manifest["splits"].items()
])
split_summary

,split,users,prefixes
0,train,47957,97122
1,valid,10317,21148
2,test,10270,20853


In [7]:
sample_path = PROJECT_ROOT / config.processed_dir / "train_prefixes.parquet"
sample = pd.read_parquet(sample_path).head(10)
sample

,user_id,split,prefix_len,num_events_total,prefix_start_ts,prefix_end_ts,event_token_ids,time_gap_ids,session_flags,next_event_token_id,label_available_retention_7d,label_retention_7d,label_available_retention_14d,label_retention_14d
0,10000235382394935150,train,50,75,1773673777,1773673783,"[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",62.0,True,0.0,False,NaN
1,10000279983408954705,train,50,406,1774705665,1774705676,"[41, 42, 43, 44, 45, 47, 46, 48, 49, 50, 136, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",407.0,False,NaN,False,NaN
2,10000279983408954705,train,100,406,1774705665,1774705707,"[41, 42, 43, 44, 45, 47, 46, 48, 49, 50, 136, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",77.0,False,NaN,False,NaN
3,10000279983408954705,train,150,406,1774705665,1774714019,"[41, 42, 43, 44, 45, 47, 46, 48, 49, 50, 136, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",74.0,False,NaN,False,NaN
4,10000413951987079100,train,50,707,1772150043,1772150050,"[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",60.0,True,0.0,True,0.0
5,10000413951987079100,train,100,707,1772150043,1772150319,"[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",9.0,True,0.0,True,0.0
6,10000413951987079100,train,150,707,1772150043,1772155288,"[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",84.0,True,0.0,True,0.0
7,10000833103973120938,train,50,83,1772487660,1772487664,"[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",66.0,True,0.0,True,0.0
8,1000084260249595813,train,50,97,1773385413,1773385421,"[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",62.0,True,0.0,True,0.0
9,10000968870629522515,train,50,4550,1773643982,1773643987,"[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",63.0,True,0.0,False,NaN


In [8]:
for split in ["train", "valid", "test"]:
    path = PROJECT_ROOT / config.processed_dir / f"{split}_prefixes.parquet"
    frame = pd.read_parquet(path)
    print("\n", split, len(frame))
    if len(frame):
        print(frame[["prefix_len", "label_available_retention_7d", "label_retention_7d", "label_available_retention_14d", "label_retention_14d"]].mean(numeric_only=True))


 train 97122
prefix_len                       87.625358
label_available_retention_7d      0.813750
label_retention_7d                0.077816
label_available_retention_14d     0.642903
label_retention_14d               0.049712
dtype: float64

 valid 21148
prefix_len                       87.906658
label_available_retention_7d      0.820456
label_retention_7d                0.076768
label_available_retention_14d     0.650747
label_retention_14d               0.049629
dtype: float64

 test 20853
prefix_len                       87.742771
label_available_retention_7d      0.813264
label_retention_7d                0.079250
label_available_retention_14d     0.650410
label_retention_14d               0.054634
dtype: float64
